# Chapter 2 - Lab 5: Financial News Agent with Evaluator-Optimizer Pattern

This version keeps the **Evaluator-Optimizer** pattern, but avoids Reuters domain filtering at the hosted-search level. In testing, the domain-restricted search returned zero structured sources even when matching Reuters articles existed. The agent now searches the public web broadly and the application filters retrieved/cited URLs to Reuters programmatically.


## 1. Install dependencies


In [ ]:
!pip install -U openai openai-agents -q


After upgrading packages in Colab, restart the runtime if requested, then continue from the next cell.


## 2. Imports and API key


In [ ]:
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Literal
from urllib.parse import urlsplit, urlunsplit

import os
from google.colab import userdata

from agents import (
    Agent,
    ItemHelpers,
    ModelSettings,
    Runner,
    TResponseInputItem,
    WebSearchTool,
)

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


## 3. Helpers to inspect real web-search sources

The final prose is not enough for debugging. These helpers inspect the structured response and extract both inline URL citations and the source URLs returned by hosted web search.


In [ ]:
@dataclass(frozen=True)
class URLCitation:
    title: str
    url: str


def get_field(obj: Any, key: str) -> Any:
    if isinstance(obj, Mapping):
        return obj.get(key)
    return getattr(obj, key, None)


def extract_url_citations(items: Sequence[Any]) -> list[URLCitation]:
    citations: list[URLCitation] = []
    seen: set[str] = set()

    for item in items:
        raw_item = get_field(item, "raw_item")
        if get_field(raw_item, "type") != "message":
            continue
        content = get_field(raw_item, "content")
        if not isinstance(content, list):
            continue
        for part in content:
            if get_field(part, "type") != "output_text":
                continue
            annotations = get_field(part, "annotations")
            if not isinstance(annotations, list):
                continue
            for annotation in annotations:
                if get_field(annotation, "type") != "url_citation":
                    continue
                url = get_field(annotation, "url")
                title = get_field(annotation, "title")
                if not isinstance(url, str) or url in seen:
                    continue
                seen.add(url)
                citations.append(URLCitation(title=title if isinstance(title, str) else url, url=url))
    return citations


def extract_web_search_source_urls(items: Sequence[Any]) -> list[str]:
    urls: list[str] = []
    seen: set[str] = set()

    for item in items:
        raw_item = get_field(item, "raw_item")
        if get_field(raw_item, "type") != "web_search_call":
            continue
        action = get_field(raw_item, "action")
        sources = get_field(action, "sources") if action else None
        if not isinstance(sources, list):
            continue
        for source in sources:
            url = get_field(source, "url")
            if not isinstance(url, str) or url in seen:
                continue
            seen.add(url)
            urls.append(url)
    return urls


def normalize_reuters_url(url: str) -> str | None:
    try:
        parsed = urlsplit(url)
    except ValueError:
        return None

    host = parsed.hostname.lower().rstrip(".") if parsed.hostname else ""
    if not (host == "reuters.com" or host.endswith(".reuters.com")):
        return None
    if parsed.scheme not in {"http", "https"}:
        return None
    path = parsed.path.rstrip("/")
    if not path:
        return None
    return urlunsplit(("https", host, path, "", ""))


def unique_reuters_urls(urls: Sequence[str]) -> list[str]:
    result: list[str] = []
    seen: set[str] = set()
    for url in urls:
        normalized = normalize_reuters_url(url)
        if normalized is None or normalized in seen:
            continue
        seen.add(normalized)
        result.append(normalized)
    return result


## 4. Searcher and evaluator agents

Important: hosted web search is **not restricted to reuters.com** here. The search can use the general web index to discover Reuters pages; after retrieval, Python keeps only direct Reuters URLs as valid evidence. Both agents use `gpt-4.1-mini` to reduce model cost.


In [ ]:
today_date = datetime.now().strftime("%Y-%m-%d")
two_days_ago = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d")

INSTRUCTIONS_NEWS_SEARCH = f"""
You are a financial news research agent.

Your job is to find genuine Reuters articles relevant to the user's request.
You have general web search. DO NOT restrict the search engine to one domain: broad web search may be needed to discover Reuters article pages.

DATE WINDOW
- Earliest allowed publication date: {two_days_ago}
- Latest allowed publication date: {today_date}

SEARCH STRATEGY
- Search the public web broadly, but only Reuters articles count as final news items.
- Use query formulations containing Reuters plus the requested company/topic.
- If one broad query is weak, search individual companies/topics separately.
- Try exact company names and terms such as Reuters OpenAI, Reuters Nvidia, Reuters Oracle, Reuters Adobe, Reuters AI infrastructure, Reuters AI investment, plus the requested geography and dates.
- Third-party pages may be used only as discovery clues. Never present them as final sources.
- A valid final source must have a direct URL whose hostname is reuters.com or a subdomain of reuters.com.
- Do not substitute stock quotes, ETF prices, index levels or generic market-data snapshots for articles.
- Do not invent headlines, dates, claims or URLs.

OUTPUT
- Respect the exact number of news items requested by the user. If no number is specified, return 5.
- For each final item include: headline, publication date, short summary and publisher Reuters.
- Include an inline web citation for every final item.
- Only finalise an item when the underlying cited page is a Reuters article in the date window.
- If fewer valid Reuters items are found after multiple searches, return only verified items and state the number found. Never fabricate missing items.
"""

web_news_searcher = Agent(
    name="web_news_searcher",
    model="gpt-4.1-mini",
    instructions=INSTRUCTIONS_NEWS_SEARCH,
    tools=[
        WebSearchTool(
            search_context_size="high",
            external_web_access=True,
        )
    ],
    model_settings=ModelSettings(
        tool_choice="required",
        response_include=["web_search_call.action.sources"],
    ),
)


@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["successful", "unsuccessful"]


INSTRUCTIONS_NEWS_EVALUATOR = f"""
You are a strict evaluator of a Reuters financial-news search result.

You receive:
1. ORIGINAL USER REQUEST
2. NEWS SUMMARY
3. REUTERS URL CITATIONS extracted from the final answer
4. REUTERS URLS RETRIEVED by web search

A result is successful only if all applicable requirements are met:
- requested number of items is present (default 5 if unspecified);
- items match the requested topic/region;
- every item has headline, publication date and short summary;
- publication dates are between {two_days_ago} and {today_date}, inclusive;
- enough genuine Reuters URLs exist in the structured evidence to support the items;
- market quotes alone do not count as news.

Do not require URLs to be manually typed in NEWS SUMMARY when they are present in structured Reuters citation/source evidence.
Do not invent requirements that were not in the original request.
Return successful only when all applicable requirements are met; otherwise return unsuccessful with specific actionable feedback.
"""

news_evaluator = Agent(
    name="news_evaluator",
    model="gpt-4.1-mini",
    instructions=INSTRUCTIONS_NEWS_EVALUATOR,
    output_type=EvaluationFeedback,
)


## 5. Evaluator-Optimizer loop with retrieval diagnostics

The diagnostics now print **all retrieved source URLs** as well as the Reuters subset. This distinguishes three cases: search is not running, search runs but never sees Reuters, or Reuters is retrieved but not used in the final answer.


In [ ]:
async def main() -> None:
    msg = input("User's request: " ).strip()

    max_iterations = 4
    latest_outline = ""
    evaluator_feedback: str | None = None

    for iteration in range(1, max_iterations + 1):
        if evaluator_feedback is None:
            search_input: list[TResponseInputItem] = [
                {"content": msg, "role": "user"}
            ]
        else:
            search_input = [
                {"content": msg, "role": "user"},
                {
                    "content": (
                        "The previous attempt failed evaluation.\n\n"
                        f"Evaluator feedback:\n{evaluator_feedback}\n\n"
                        "Perform a completely NEW web search. Broaden the search wording and search individual companies/topics separately. "
                        "Do not restrict the search engine by domain; instead find direct Reuters article pages through the public web index. "
                        "Do not reuse unsupported claims from the previous answer."
                    ),
                    "role": "user",
                },
            ]

        news_searcher_result = await Runner.run(web_news_searcher, search_input)
        latest_outline = ItemHelpers.text_message_outputs(news_searcher_result.new_items)

        citations = extract_url_citations(news_searcher_result.new_items)
        all_retrieved_urls = extract_web_search_source_urls(news_searcher_result.new_items)
        cited_reuters_urls = unique_reuters_urls([c.url for c in citations])
        retrieved_reuters_urls = unique_reuters_urls(all_retrieved_urls)

        print("\n\033[92m" + f"************************** NEWS SEARCH {iteration} **************************" + "\033[0m")
        print(latest_outline)

        print("\n\033[93m************************** WEB SEARCH DIAGNOSTICS **************************\033[0m")
        print(f"All URLs retrieved by web search: {len(all_retrieved_urls)}")
        for url in all_retrieved_urls[:20]:
            print("  SOURCE:", url)
        print(f"Reuters citations in final response: {len(cited_reuters_urls)}")
        for url in cited_reuters_urls:
            print("  CITED REUTERS:", url)
        print(f"Reuters URLs retrieved by web search: {len(retrieved_reuters_urls)}")
        for url in retrieved_reuters_urls:
            print("  RETRIEVED REUTERS:", url)

        evaluator_input = (
            f"ORIGINAL USER REQUEST:\n{msg}\n\n"
            f"NEWS SUMMARY:\n{latest_outline}\n\n"
            "REUTERS URL CITATIONS:\n" + ("\n".join(cited_reuters_urls) if cited_reuters_urls else "NONE") + "\n\n"
            "REUTERS URLS RETRIEVED BY WEB SEARCH:\n" + ("\n".join(retrieved_reuters_urls) if retrieved_reuters_urls else "NONE")
        )

        print("\n\033[92m************************** RUNNING EVALUATION **************************\033[0m")
        news_evaluator_result = await Runner.run(news_evaluator, evaluator_input)
        result: EvaluationFeedback = news_evaluator_result.final_output
        print(f"\033[94mEvaluator score: {result.score}\033[0m")
        print(f"\033[94mEvaluator feedback: {result.feedback}\033[0m")

        if result.score == "successful":
            print("\033[92mEvaluation successful ==> stopping iteration.\033[0m")
            break

        evaluator_feedback = result.feedback
        if iteration == max_iterations:
            print("\033[91mReached max_iterations ==> stopping iteration.\033[0m")

    print("\n\033[92m************************** FINAL NEWS SET **************************\033[0m")
    print(latest_outline)


## 6. Run

Example:

`Give me the latest 5 Reuters articles from the last 2 days about OpenAI, Nvidia, Oracle, Adobe, AI infrastructure or AI investment in the United States.`


In [ ]:
await main()
